# 03 · Uso del modelo final

Recorrido de una predicción con los artefactos de `modelos/`: carga del modelo, construcción de las variables de la última emisión disponible con el código del pipeline, probabilidad y estado de alerta del producto.

- **Datos.** Producto del canal (`dvc pull`), del que se toman las quince columnas del panel; el código recalcula el canal y las variables solo con semanas ≤ t.
- **Modelos.** `modelos/<código>.joblib` (copia de `data/processed/forecasting/prueba/modelos_finales/`).
- **Alcance.** No escribe archivos y no usa MLflow: la API cargará los mismos modelos desde el registro de modelos de MLflow (`DengueAlertas_<Municipio>`).

In [1]:
import os
import sys
from pathlib import Path

# Rutas relativas a la raíz del repositorio, tanto si el cuaderno se abre desde notebooks/ como desde la raíz.
ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "forecasting.yaml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "configs" / "forecasting.yaml").is_file():
    raise RuntimeError("El cuaderno se ejecuta desde la raíz del repositorio o desde notebooks/.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import json
from datetime import timedelta

import joblib
import pandas as pd
import pyarrow.compute as pc

from src.epidemiology.validation import SOURCE_SCHEMA
from src.forecasting.config import load_config
from src.forecasting.dataset import FEATURE_SETS, FEATURES, build_features, load_channel_product
from src.forecasting.events import CONTINUIDAD, emission_positive, product_alert_states
from src.forecasting.guard import load_frozen_recipe
from src.forecasting.models import predict_positive

config = load_config()
receta = load_frozen_recipe("configs/receta_congelada.yaml", channel_version=config.datos.version_dvc_canal)
manifiesto = json.loads(Path("modelos/manifiesto.json").read_text(encoding="utf-8"))
assert manifiesto["receta_hash"] == receta["hash_sha256"], "Los modelos no corresponden a la receta congelada."

TAU, K = receta["decision"]["tau"], receta["decision"]["k"]
REGLA = config.producto.regla_alerta   # regla del producto (D10.1); la evaluación preregistrada usa decision.regla_emision
H = config.horizonte.h
print(f"Receta {receta['hash_sha256'][:8]}…: τ = {TAU}, k = {K}, h = {H}; regla del producto: {REGLA}")

Receta de74d4ab…: τ = 0.05, k = 2, h = 8; regla del producto: umbral_fuera_de_epidemia


## 1. Cargar un modelo

Cada archivo es un `Pipeline` de scikit-learn: estandarización de las cinco variables numéricas, dos indicadores de zona (referencia: seguridad) y la regresión logística L2.

In [3]:
CODIGO = "50001"   # Villavicencio
modelo = joblib.load(f"modelos/{CODIGO}.joblib")
info = manifiesto["modelos"][CODIGO]
print(f"{CODIGO}: {info['familia']} {info['candidato']} {info['hiperparametros']}, variables {info['conjunto_variables']}, "
      f"{info['filas']} filas, última emisión de entrenamiento {info['ultima_emision']}")
coeficientes = pd.Series(modelo[-1].coef_[0], index=modelo[0].get_feature_names_out(), name="Coeficiente")
coeficientes["intercepto"] = modelo[-1].intercept_[0]
coeficientes.round(3).to_frame()

50001: logistica logistica_05 {'C': 3.0, 'max_iter': 5000}, variables sin_clima, 711 filas, última emisión de entrenamiento 2025-10-27


,Coeficiente
estandarizar__posicion_4s,3.447
estandarizar__crecimiento_4s,0.310
estandarizar__cambio_umbral,-1.010
estandarizar__sem_sin,0.614
estandarizar__sem_cos,0.718
zona__zona_t_alerta,0.917
zona__zona_t_epidemia,1.094
intercepto,-4.390


## 2. Construir las variables de la última emisión disponible

La última semana completa de la base es la última emisión posible. El estado de alerta depende de las emisiones anteriores (k = 2), así que se construyen todas las emisiones de la ventana del modelo final: desde la semana siguiente a su última emisión de entrenamiento hasta la última semana completa. `build_features` recibe solo las quince columnas del panel y recalcula el canal en cada t.

In [4]:
producto = load_channel_product(config.datos.canal, expected_md5=config.datos.version_dvc_canal)
panel = producto.select(SOURCE_SCHEMA.names)
ultima = pc.max(pc.filter(panel["week_start"], pc.invert(panel["partial_week"]))).as_py()   # última semana completa
primera = pd.Timestamp(info["ultima_emision"]).date() + timedelta(weeks=1)
semanas = [primera + timedelta(weeks=i) for i in range((ultima - primera).days // 7 + 1)]
codigos = sorted(manifiesto["modelos"])

variables = build_features(panel, [(c, t) for c in codigos for t in semanas], h=H,
                           canal=config.canal, features=config.features)
print(f"Emisiones: {semanas[0]} a {semanas[-1]} ({len(semanas)} por municipio); semana objetivo de la última: "
      f"{ultima + timedelta(weeks=H)}")
print(f"El modelo usa {len(FEATURE_SETS[info['conjunto_variables']])} de las {len(FEATURES)} variables construidas.")
variables[variables["t"] == ultima][["municipality_code", "t", *FEATURE_SETS["sin_clima"]]].round(4)

Emisiones: 2025-11-03 a 2025-12-22 (8 por municipio); semana objetivo de la última: 2026-02-16
El modelo usa 6 de las 15 variables construidas.


,municipality_code,t,zona_t,posicion_4s,crecimiento_4s,cambio_umbral,sem_sin,sem_cos
7,08001,2025-12-22,seguridad,-1.3506,0.3933,-1.4363,0.6612,0.7502
15,08758,2025-12-22,seguridad,-1.9423,0.1542,-1.1955,0.6612,0.7502
23,41001,2025-12-22,seguridad,-2.7300,-0.9163,0.8359,0.6612,0.7502
31,50001,2025-12-22,epidemia,0.3344,0.5167,1.4636,0.6612,0.7502
39,54001,2025-12-22,seguridad,-1.1370,-0.2231,-0.0307,0.6612,0.7502
47,76001,2025-12-22,seguridad,-3.1252,0.0800,0.5157,0.6612,0.7502


## 3. Probabilidad y estado de alerta del producto

Para cada municipio: p̂ con su modelo y, con la secuencia de emisiones, el estado de `events.product_alert_states` con la regla del producto (B01): la emisión es positiva si p̂ ≥ τ y la zona observada en t no es epidemia.

In [5]:
resultados = []
for codigo in codigos:
    modelo_m = joblib.load(f"modelos/{codigo}.joblib")
    filas = variables[variables["municipality_code"] == codigo].sort_values("t")
    p = predict_positive(modelo_m, filas[list(FEATURES)])
    zonas = filas["zona_t"].tolist()
    estados = product_alert_states(filas["t"].tolist(), p, zonas, tau=TAU, k=K, rule=REGLA)
    positivas = emission_positive(REGLA, p, zonas, tau=TAU)
    racha = 0
    for t, zona, prob, positiva, estado in zip(filas["t"], zonas, p, positivas, estados):
        racha = racha + 1 if positiva else 0
        resultados.append({
            "municipio": codigo, "semana_emision": t, "semana_objetivo": t + timedelta(weeks=H), "zona_t": zona,
            "probabilidad": round(float(prob), 4), "p_supera_tau": bool(prob >= TAU), "estado_alerta": estado,
            "etiqueta_probabilidad": ("seguir_en_zona_epidemica" if estado == CONTINUIDAD
                                      else "entrar_en_zona_epidemica"),
            "emisiones_positivas_consecutivas": racha,
        })
resultados = pd.DataFrame(resultados)
resultados[resultados["semana_emision"] == ultima].set_index("municipio")

,semana_emision,semana_objetivo,zona_t,probabilidad,p_supera_tau,estado_alerta,etiqueta_probabilidad,emisiones_positivas_consecutivas
municipio,,,,,,,,
08001,2025-12-22,2026-02-16,seguridad,0.0296,False,sin_alerta,entrar_en_zona_epidemica,0
08758,2025-12-22,2026-02-16,seguridad,0.0468,False,sin_alerta,entrar_en_zona_epidemica,0
41001,2025-12-22,2026-02-16,seguridad,0.0001,False,sin_alerta,entrar_en_zona_epidemica,0
50001,2025-12-22,2026-02-16,epidemia,0.4158,True,continuidad_epidemica,seguir_en_zona_epidemica,0
54001,2025-12-22,2026-02-16,seguridad,0.0160,False,sin_alerta,entrar_en_zona_epidemica,0
76001,2025-12-22,2026-02-16,seguridad,0.0002,False,sin_alerta,entrar_en_zona_epidemica,0


Trayectoria del municipio elegido en la ventana del modelo final. La primera emisión de la ventana no tiene emisión anterior en esta secuencia, así que su estado y su racha no cuentan la emisión del 2025-10-27, que pertenece al entrenamiento del modelo final. El estado de la última emisión no depende de ello: con k = 2 basta con las dos últimas emisiones.

In [6]:
resultados[resultados["municipio"] == CODIGO].set_index("semana_emision")

,municipio,semana_objetivo,zona_t,probabilidad,p_supera_tau,estado_alerta,etiqueta_probabilidad,emisiones_positivas_consecutivas
semana_emision,,,,,,,,
2025-11-03,50001,2025-12-29,alerta,0.2340,True,senal_en_observacion,entrar_en_zona_epidemica,1
2025-11-10,50001,2026-01-05,alerta,0.0537,True,alerta_inicio_confirmada,entrar_en_zona_epidemica,2
2025-11-17,50001,2026-01-12,alerta,0.1128,True,alerta_inicio_confirmada,entrar_en_zona_epidemica,3
2025-11-24,50001,2026-01-19,alerta,0.1210,True,alerta_inicio_confirmada,entrar_en_zona_epidemica,4
2025-12-01,50001,2026-01-26,epidemia,0.1262,True,continuidad_epidemica,seguir_en_zona_epidemica,0
2025-12-08,50001,2026-02-02,epidemia,0.2123,True,continuidad_epidemica,seguir_en_zona_epidemica,0
2025-12-15,50001,2026-02-09,epidemia,0.4173,True,continuidad_epidemica,seguir_en_zona_epidemica,0
2025-12-22,50001,2026-02-16,epidemia,0.4158,True,continuidad_epidemica,seguir_en_zona_epidemica,0


## 4. Cómo se lee el resultado

- **Probabilidad.** `probabilidad` es siempre p̂ = P(zona epidémica en la semana objetivo t + 8), no la probabilidad de un brote ni un número de casos. Se etiqueta `seguir_en_zona_epidemica` cuando el municipio ya está en zona epidémica en t (`continuidad_epidemica`) y `entrar_en_zona_epidemica` en los otros tres estados. No hay bandas de riesgo (contrato §9.5).
- **Estados** (`api_salida_d10.md` §2):
  - `sin_alerta`: fuera de zona epidémica y p̂ < τ;
  - `senal_en_observacion`: primera emisión positiva de una racha (menos de k = 2);
  - `alerta_inicio_confirmada`: k = 2 emisiones positivas consecutivas fuera de zona epidémica;
  - `continuidad_epidemica`: la zona observada en t ya es epidemia; la regla del producto no emite alerta y la probabilidad se lee como probabilidad de seguir.
- **Umbral.** τ = 0,05 se eligió en desarrollo con el criterio del contrato §7.5: la mayor detección de inicios con falsas alarmas por municipio-año no mayores que las del canal. `p_supera_tau` muestra la emisión de la regla preregistrada (p̂ ≥ τ), que difiere de la del producto solo en zona epidémica.
- **Validez.** El desempeño medido en prueba vale para τ y k congelados. Las predicciones de esta ventana no tienen todavía verdad observada: sus semanas objetivo son posteriores a la base. Para semanas anteriores, la vista de historial usa las predicciones fuera de muestra precalculadas, no el modelo final (contrato §9.4).